In [3]:
import numpy as np

import jax
import jax.numpy as jnp
from jax import lax
import matplotlib.pyplot as plt


In [4]:
# ----------------------------
#  Fixed-envelope, phase-only pulse optimization
# ----------------------------

# Basis: |g> = [1,0], |e> = [0,1]
# Hamiltonian (rotating frame, hbar = 1):
# H(t) = 1/2 [Omega_x(t) sigma_x + Omega_y(t) sigma_y + Delta sigma_z]
#
# The envelope is fixed to a Lorentzian.
# Only the phase bins theta_j are optimized.
#
# The same phase sequence is evaluated at several pulse areas A in `areas`,
# e.g. A = pi, 3pi, 5pi, to encourage "power narrowing".

I2 = jnp.array([[1.0, 0.0], [0.0, 1.0]], dtype=jnp.complex64)
SX = jnp.array([[0.0, 1.0], [1.0, 0.0]], dtype=jnp.complex64)
SY = jnp.array([[0.0, -1.0j], [1.0j, 0.0]], dtype=jnp.complex64)
SZ = jnp.array([[1.0, 0.0], [0.0, -1.0]], dtype=jnp.complex64)

GROUND = jnp.array([1.0 + 0.0j, 0.0 + 0.0j], dtype=jnp.complex64)


In [5]:
def make_lorentzian_envelope(num_bins: int, total_time: float, gamma: float, power: float=1):
    """Piecewise-constant Lorentzian envelope normalized to unit area.

    Returns:
        t_mid: midpoint times, shape (N,)
        env:   normalized envelope, shape (N,), with sum(env) * dt = 1
        dt:    bin duration
        power: power of the raw Lorentzian shape (< 1 for more narrowing)
    """
    dt = total_time / num_bins
    t_mid = (jnp.arange(num_bins) + 0.5) * dt - 0.5 * total_time
    raw = 1.0 / (1.0 + (t_mid / gamma) ** 2) ** power
    env = raw / (jnp.sum(raw) * dt)
    return t_mid, env, dt


In [6]:
def segment_unitary(delta: float, omega: float, phase: float, dt: float):
    """Exact 2x2 propagator for one piecewise-constant segment."""
    hx = omega * jnp.cos(phase)
    hy = omega * jnp.sin(phase)
    hz = delta

    hnorm = jnp.sqrt(hx * hx + hy * hy + hz * hz + 1e-30)
    alpha = 0.5 * hnorm * dt

    hdot_sigma = hx * SX + hy * SY + hz * SZ
    return jnp.cos(alpha) * I2 - 1.0j * jnp.sin(alpha) * (hdot_sigma / hnorm)


In [7]:
def final_state(theta, env, dt, delta, area):
    """Final state for one detuning and one pulse area.

    theta: unconstrained phases, shape (N,)
    env:   fixed envelope with integral 1
    area:  pulse area A, so Omega(t) = A * env(t)
    """
    omegas = area * env

    def step(psi, x):
        omega_j, theta_j = x
        Uj = segment_unitary(delta, omega_j, theta_j, dt)
        psi = Uj @ psi
        return psi, None

    psi_f, _ = lax.scan(step, GROUND, (omegas, theta))
    return psi_f


In [8]:
def excited_population(theta, env, dt, delta, area):
    psi_f = final_state(theta, env, dt, delta, area)
    return jnp.real(jnp.conj(psi_f[1]) * psi_f[1])


In [9]:
# Vectorize over detuning and pulse area
excited_population_vs_detuning = jax.vmap(
    excited_population, in_axes=(None, None, None, 0, None)
)
population_grid = jax.vmap(
    excited_population_vs_detuning, in_axes=(None, None, None, None, 0)
)


In [10]:
def trapz_uniform(y, x):
    dx = x[1] - x[0]
    return dx * (0.5 * y[0] + jnp.sum(y[1:-1]) + 0.5 * y[-1])


In [11]:
def smooth_abs(x, eps=1e-12):
    return jnp.sqrt(x * x + eps)


def smooth_ramp(x, tau=0.03):
    # smooth approximation to max(x, 0)
    return tau * jax.nn.softplus(x / tau)


def smooth_cap(x, xmax=8.0, tau=0.03):
    # smooth approximation to min(x, xmax)
    return xmax - tau * jax.nn.softplus((xmax - x) / tau)


def make_offres_mask_and_weight(detunings, delta_c=0.20, delta0=0.75, tau=0.03, max_log_weight=8.0):
    """
    mask(Δ): ~0 near resonance, ~1 off resonance
    weight(Δ): grows smoothly/exponentially with |Δ|
    """
    ad = smooth_abs(detunings)

    # smooth exclusion of the central resonance region
    mask = jax.nn.sigmoid((ad - delta_c) / tau)

    # smooth exponential growth away from resonance
    x = smooth_ramp((ad - delta_c) / delta0, tau=tau)
    x = smooth_cap(x, xmax=max_log_weight, tau=tau)
    weight = jnp.exp(x)
    weight = weight / jnp.mean(weight)

    return mask, weight


In [12]:
def phase_smoothness(theta):
    """Periodic smoothness penalty: small if consecutive phases are close on S^1."""
    dtheta = theta[1:] - theta[:-1]
    return jnp.mean(1.0 - jnp.cos(dtheta))


In [13]:
def build_loss_fn(
    env,
    dt,
    detunings,
    areas,
    ratio_weight=1.0,
    center_weight=0.5,
    order_weight=0.2,
    smoothness_weight=1e-3,
    delta_c=0.20,
    delta0=0.75,
    tau=0.03,
    eps=1e-3,
    alpha=2.0,
    max_ratio_exponent=8.0,
    compare_weights=None,
):
    """
    Returns a scalar loss(theta) and aux metrics.

    Interpretation:
      - areas[0] is the reference pulse (usually π)
      - each higher-area pulse is penalized when its off-resonant population
        is nonzero relative to the π-pulse profile

    Main term:
      mean_Δ[ mask(Δ) * weight(Δ) * (exp(alpha * R_k(Δ)) - 1) ]
      with R_k(Δ) = P_k(Δ) / (P_pi(Δ) + eps * P_pi(0))
    """
    i0 = int(jnp.argmin(jnp.abs(detunings)))
    mask, det_weight = make_offres_mask_and_weight(
        detunings,
        delta_c=delta_c,
        delta0=delta0,
        tau=tau,
    )

    n_cmp = int(areas.shape[0]) - 1
    if compare_weights is None:
        compare_weights = jnp.ones((n_cmp,))
    else:
        compare_weights = jnp.asarray(compare_weights)
        if compare_weights.shape != (n_cmp,):
            raise ValueError(f"compare_weights must have shape ({n_cmp},)")

    def loss_fn(theta):
        pops = population_grid(theta, env, dt, detunings, areas)  # (n_areas, n_detunings)

        P_ref = pops[0]                # usually P_pi(Δ)
        P_ref0 = P_ref[i0]             # usually P_pi(0)
        denom_floor = eps * (P_ref0 + 1e-8)

        ratio_losses = []
        weighted_mean_ratios = []
        pointwise_ratios = []

        for k in range(1, pops.shape[0]):
            Pk = pops[k]

            # ratio to suppress off resonance
            Rk = Pk / (P_ref + denom_floor)

            # smooth cap before exp to avoid numerical blow-up
            z = smooth_cap(alpha * Rk, xmax=max_ratio_exponent, tau=tau)

            pointwise_penalty = mask * det_weight * jnp.expm1(z)
            loss_k = jnp.mean(pointwise_penalty)

            ratio_losses.append(loss_k)
            weighted_mean_ratios.append(jnp.mean(mask * det_weight * Rk))
            pointwise_ratios.append(Rk)

        ratio_losses = jnp.stack(ratio_losses)                       # one value for 3π, 5π, ...
        weighted_mean_ratios = jnp.stack(weighted_mean_ratios)
        pointwise_ratios = jnp.stack(pointwise_ratios)               # shape (n_cmp, n_detunings)

        ratio_term = jnp.sum(compare_weights * ratio_losses) / jnp.sum(compare_weights)

        # keep the on-resonance excitation high for all odd-π pulses
        center_pops = pops[:, i0]
        center_loss = jnp.mean((1.0 - center_pops) ** 2)

        # encourage stronger narrowing as pulse area increases:
        # penalty is small when ratio_losses[1] <= ratio_losses[0], etc.
        if ratio_losses.shape[0] > 1:
            ordering_penalty = jnp.mean(
                jax.nn.softplus(10.0 * (ratio_losses[1:] - ratio_losses[:-1])) / 10.0
            )
        else:
            ordering_penalty = 0.0

        smooth_pen = phase_smoothness(theta)

        loss = (
            ratio_weight * ratio_term
            + center_weight * center_loss
            + order_weight * ordering_penalty
            + smoothness_weight * smooth_pen
        )

        aux = {
            "ratio_losses": ratio_losses,
            "weighted_mean_ratios": weighted_mean_ratios,
            "pointwise_ratios": pointwise_ratios,
            "center_pops": center_pops,
            "ordering_penalty": ordering_penalty,
            "smooth_penalty": smooth_pen,
            "mask": mask,
            "detuning_weight": det_weight,
            "populations": pops,
        }
        return loss, aux

    return loss_fn

In [18]:
def optimize_phases(
    theta0,
    loss_fn,
    num_steps=800,
    learning_rate=5e-2,
    beta1=0.9,
    beta2=0.999,
    eps=1e-8,
    print_every=50,
):
    """Simple Adam optimizer in pure JAX + Python."""
    value_and_grad_fn = jax.jit(jax.value_and_grad(loss_fn, has_aux=True))

    theta = theta0
    m = jnp.zeros_like(theta)
    v = jnp.zeros_like(theta)

    history = {
        "loss": [],
        "weighted_mean_ratios": [],
        "pointwise_ratios": [],
        "center_pops": [],
        "ordering_penalty": [],
        "smooth_penalty": [],
        "mask": [],
        "detuning_weight": [],
        "populations": [],
    }

    for step in range(1, num_steps + 1):
        (loss, aux), grad = value_and_grad_fn(theta)

        m = beta1 * m + (1.0 - beta1) * grad
        v = beta2 * v + (1.0 - beta2) * (grad * grad)

        m_hat = m / (1.0 - beta1**step)
        v_hat = v / (1.0 - beta2**step)

        theta = theta - learning_rate * m_hat / (jnp.sqrt(v_hat) + eps)

        history["loss"].append(float(loss))
        history["weighted_mean_ratios"].append(np.array(aux["weighted_mean_ratios"]))
        history["pointwise_ratios"].append(np.array(aux["pointwise_ratios"]))
        history["center_pops"].append(np.array(aux["center_pops"]))
        history["ordering_penalty"].append(float(aux["ordering_penalty"]))
        history["smooth_penalty"].append(float(aux["smooth_penalty"]))
        history["mask"].append(aux["mask"])
        history["detuning_weight"].append(np.array(aux["detuning_weight"]))
        history["populations"].append(np.array(aux["populations"]))
        
        if step == 1 or step % print_every == 0:
            ratio_losses_str = ", ".join([f"{w:.5f}" for w in np.array(aux["ratio_losses"])])
            peaks_str = ", ".join([f"{p:.5f}" for p in np.array([3, 5])])
            print(
                f"step={step:4d}  loss={float(loss): .6f}  "
                f"Ratio losses=[{ratio_losses_str}]  peaks=[{[str(p)+"pi" for p in peaks_str]}]"
            )

    return theta, history


In [19]:
def evaluate_solution(theta, env, dt, detunings, areas, sharpness=40.0):
    pops = population_grid(theta, env, dt, detunings, areas)
    widths = []
    peaks = []
    for k in range(areas.shape[0]):
        width_k, peak_k, _, _ = smooth_fwhm_about_zero(
            detunings, pops[k], threshold=0.5, sharpness=sharpness
        )
        widths.append(width_k)
        peaks.append(peak_k)
    return pops, jnp.stack(widths), jnp.stack(peaks)


In [20]:
def main():
    # ----------------------------
    # User-tunable settings
    # ----------------------------
    num_bins = 100                 # number of phase bins to optimize
    total_time = 16.0              # total pulse duration
    gamma = 0.9                   # Lorentzian width parameter of the envelope
    detuning_max = 4.0
    n_detunings = 401             # use an odd number so Delta=0 is included exactly
    areas = jnp.array([jnp.pi, 3.0 * jnp.pi, 5.0 * jnp.pi], dtype=jnp.float32)

    # Loss hyperparameters
    width_weight = 1.0
    peak_weight = 0.8
    side_weight = 0.05
    order_weight = 0.2
    smoothness_weight = 5e-4
    sharpness = 50.0

    # Optimization hyperparameters
    num_steps = 1000
    learning_rate = 5e-3
    seed = 42

    # ----------------------------
    # Build fixed Lorentzian envelope
    # ----------------------------
    t_mid, env, dt = make_lorentzian_envelope(num_bins, total_time, gamma)
    detunings = jnp.linspace(-detuning_max, detuning_max, n_detunings)

    # ----------------------------
    # Initial guess for phases
    # ----------------------------
    key = jax.random.PRNGKey(seed)
    theta0 = 0.05 * jax.random.normal(key, shape=(num_bins,))

    # ----------------------------
    # Build and optimize objective
    # ----------------------------
    loss_fn = build_loss_fn(
        env=env,
        dt=dt,
        detunings=detunings,
        areas=areas,
        ratio_weight=1.0,
        center_weight=0.5,
        order_weight=0.2,
        smoothness_weight=1e-3,
        delta_c=0.20,
        delta0=0.75,
        tau=0.03,
        eps=1e-3,
        alpha=2.0,
        max_ratio_exponent=8.0,
        compare_weights=None,
    )


    theta_opt, history = optimize_phases(
        theta0,
        loss_fn,
        num_steps=num_steps,
        learning_rate=learning_rate,
        print_every=50,
    )

    phases_wrapped = jnp.mod(theta_opt, 2.0 * jnp.pi)
    pops, widths, peaks = evaluate_solution(theta_opt, env, dt, detunings, areas, sharpness=sharpness)

    print("\nFinal wrapped phases (rad):")
    print(np.array(phases_wrapped))

    print("\nFinal smooth widths:")
    for area, width, peak in zip(np.array(areas), np.array(widths), np.array(peaks)):
        print(f"area={area/np.pi:.1f}π   width={width:.6f}   P(0)={peak:.6f}")

    # ----------------------------
    # Plots
    # ----------------------------
    plt.figure(figsize=(7, 4))
    plt.plot(np.array(t_mid), np.array(env))
    plt.xlabel("time")
    plt.ylabel("normalized Lorentzian envelope")
    plt.title("Fixed envelope")
    plt.tight_layout()

    plt.figure(figsize=(7, 4))
    plt.plot(np.array(phases_wrapped), lw=1.5)
    plt.xlabel("phase bin")
    plt.ylabel("phase [rad]")
    plt.title("Optimized phase sequence")
    plt.tight_layout()

    plt.figure(figsize=(7, 4))
    for k, area in enumerate(np.array(areas)):
        plt.plot(np.array(detunings), np.array(pops[k]), label=f"{area/np.pi:.0f}π")
    plt.xlabel("detuning Δ")
    plt.ylabel("excited-state population")
    plt.title("Lineshapes after optimization")
    plt.legend()
    plt.tight_layout()

    plt.figure(figsize=(7, 4))
    plt.plot(history["loss"])
    plt.xlabel("optimization step")
    plt.ylabel("loss")
    plt.title("Training curve")
    plt.tight_layout()

    plt.show()


In [17]:
main()

step=   1  loss= 4.524879  widths=[0.36266, 6.30868]  peaks=[9.42478, 15.70796]
step=  50  loss= 0.147968  widths=[0.09344, 0.15943]  peaks=[9.42478, 15.70796]
step= 100  loss= 0.134603  widths=[0.07894, 0.14672]  peaks=[9.42478, 15.70796]
step= 150  loss= 0.133260  widths=[0.07768, 0.14533]  peaks=[9.42478, 15.70796]
step= 200  loss= 0.132129  widths=[0.07670, 0.14411]  peaks=[9.42478, 15.70796]
step= 250  loss= 0.130960  widths=[0.07610, 0.14261]  peaks=[9.42478, 15.70796]
step= 300  loss= 0.129642  widths=[0.07577, 0.14072]  peaks=[9.42478, 15.70796]
step= 350  loss= 0.128077  widths=[0.07564, 0.13831]  peaks=[9.42478, 15.70796]
step= 400  loss= 0.126168  widths=[0.07569, 0.13525]  peaks=[9.42478, 15.70796]
step= 450  loss= 0.123827  widths=[0.07594, 0.13137]  peaks=[9.42478, 15.70796]
step= 500  loss= 0.121004  widths=[0.07644, 0.12656]  peaks=[9.42478, 15.70796]
step= 550  loss= 0.117735  widths=[0.07725, 0.12082]  peaks=[9.42478, 15.70796]
step= 600  loss= 0.114169  widths=[0.078

NameError: name 'smooth_fwhm_about_zero' is not defined